In [4]:
import sys
import numpy as np
import pandas as pd
import pathlib
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
from pprint import pprint
import yaml
from tqdm import tqdm

# notebook上で.pyファイルを設定しなおして、リロードする場合
import importlib

#- sys.modules に一度読み込まれたモジュールは、再度 import してもキャッシュから取得されます。
#- importlib.reload() はそのキャッシュを明示的に更新する方法
# .pyファイルのリロードする場合
#importlib.reload(ecg_reader)

#### Set Project Root Directory

In [5]:
#config.yamlを配置したフォルダをルートフォルダと設定する
def find_project_root(marker: str = "config.yaml") -> Path:
    path = Path.cwd()
    while not (path / marker).exists() and path != path.parent:
        path = path.parent
    return path

project_root = find_project_root()
print("RootDirectory Path:",project_root)

# system Pathに追加
sys.path.append(str(project_root))
#pprint(sys.path)
#project_root = Path.cwd().parent で設定するとuserName/projectsがparaentsフォルダ賭して認識されるため上記方法で固定とする

### フォルダ構成の確認
directory_file_list = list(project_root.glob("*"))
pprint(directory_file_list)

RootDirectory Path: /Users/yohe/Desktop/ECG-Analysis
[PosixPath('/Users/yohe/Desktop/ECG-Analysis/analysis_test.ipynb'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/config.yaml'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/README.md'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/.git'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/MEMO.md'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/notebooks'),
 PosixPath('/Users/yohe/Desktop/ECG-Analysis/src')]


In [6]:
from src.data_utils.ecg_reader import load_table_as_pandas, load_ecg_metadata_by_id, extract_lead_features, plot_ecg_waveform
from src.system_utils import detect_os, windows_to_wsl_path, check_config_paths
from src.config_loader import DataConfig

### Set CONFIG

In [9]:
# OSを判定し、WSLの環境ではWindowsのディレクトリにアクセス可能なようにRootDirectoryを変更
detected_os = detect_os()
print("System Environment:", detected_os)

#wsl
#set_root_directory = r"C:\Users\yohei\Desktop\physionet-ecg-image-digitization"
#mac
set_root_directory = "/Users/yohe/Desktop/physionet-ecg-image-digitization"

if detected_os == "wsl":
    ECG_DATASET_ROOT = windows_to_wsl_path(set_root_directory)
else:
    ECG_DATASET_ROOT = Path(set_root_directory)

print("Project Root:", ECG_DATASET_ROOT)

System Environment: mac
Project Root: /Users/yohe/Desktop/physionet-ecg-image-digitization


In [10]:
# datareader用の設定
CONFIG = DataConfig(config_yaml_path=project_root/"config.yaml", 
                    root=ECG_DATASET_ROOT)
#CONFIGのパラメータを確認
pprint(CONFIG.__dict__)

# CONFIG内のパスが存在するか確認
check_config_paths(CONFIG)

{'sample_submission_csv_path': PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/sample_submission.parquet'),
 'test_csv_path': PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/test.csv'),
 'test_data_dir': PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/test'),
 'train_csv_path': PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train.csv'),
 'train_data_dir': PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train')}
すべてのパスが存在します


### read dataframe

In [11]:
train_df = load_table_as_pandas(path=CONFIG.train_csv_path, verbose=True)
test_df = load_table_as_pandas(path=CONFIG.test_csv_path, verbose=True)
sample_submission = load_table_as_pandas(path=CONFIG.sample_submission_csv_path, verbose=True)

Datatype is CSV
Datatype is CSV
Datatype is Parquet


In [15]:
### サンプルデータで出力テストをする

In [22]:
sample_id = 11842146
sample_data_dir = Path(CONFIG.train_data_dir / str(sample_id))
print("Sample Data Directory:", sample_data_dir)
list(sample_data_dir.glob("*"))

Sample Data Directory: /Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146


[PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0003.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0001.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0004.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0010.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0011.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0005.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0006.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0012.png'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146.csv'),
 PosixPath('/Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146/11842146-0009.pn

### CSVデータ処理

In [23]:
import neurokit2 as nk

In [26]:
### リードデータを抽出する
def extract_lead_signal(ecg_df: pd.DataFrame, lead: str = "I") -> np.ndarray:
    """
    指定されたリードの波形を抽出し、NaNを除去して返す。
    
    Parameters:
        ecg_df (pd.DataFrame): ECGの波形データ（CSV読み込み済み）
        lead (str): 抽出対象のリード名（例："I", "II", "V1"など）

    Returns:
        np.ndarray: NaN除去済みの波形データ
    """
    if lead not in ecg_df.columns:
        raise ValueError(f"リード '{lead}' は存在しません。利用可能なリード: {list(ecg_df.columns)}")

    signal = pd.to_numeric(ecg_df[lead], errors="coerce").dropna().values

    if len(signal) == 0:
        raise ValueError(f"リード '{lead}' に有効な波形データが存在しません。")

    return signal

In [32]:
sample_id = 11842146

# set directory
sample_data_dir = Path(CONFIG.train_data_dir / str(sample_id))
print("Sample Data Directory:", sample_data_dir)

# set csv data path
sample_csv_path = list(sample_data_dir.glob("*.csv"))[0]

ecg_df = pd.read_csv(sample_csv_path)
lead_signal = extract_lead_signal(ecg_df=ecg_df, lead="I")
print("信号長:", len(lead_signal))

Sample Data Directory: /Users/yohe/Desktop/physionet-ecg-image-digitization/train/11842146
信号長: 2500


### 1. R peakの検出

In [39]:

def detect_r_peaks(signal: np.ndarray, fs: int = 1000) -> dict:
    """
    ECG波形からRピークを検出する関数（NeuroKit2使用）

    Parameters:
        signal (np.ndarray): NaN除去済みのECG波形
        fs (int): サンプリング周波数（Hz）

    Returns:
        dict: 以下のキーを含む辞書
            - "rpeaks": Rピーク位置（int型のnumpy配列）
            - "signals": NeuroKit2の信号構造
            - "info": Rピーク検出情報（辞書）
    """
    if len(signal) < 100:
        raise ValueError("信号が短すぎます。最低100サンプル以上必要です。")

    # Rピーク検出（NeuroKit2）
    signals, info = nk.ecg_peaks(signal, sampling_rate=fs)

    # Rピーク位置を整数型で抽出
    rpeaks = pd.Series(info["ECG_R_Peaks"]).dropna().astype(int).values

    return {
        "rpeaks": rpeaks,
        "signals": signals,
        "info": info
    }

In [44]:
#- Rピーク数が 5 未満 → nk.ecg_delineate() は失敗
#- Rピーク間隔がすべて同じ → 周期性が強すぎて失敗
signal = extract_lead_signal(ecg_df, lead="I")
print("shape of signal:",signal.shape)

r_result = detect_r_peaks(signal, fs=1000)
rpeaks = r_result["rpeaks"]
print("rpeaks:", rpeaks)
print("Rピーク数:", len(rpeaks))
print("Rピーク間隔:", np.diff(rpeaks))


shape of signal: (2500,)
rpeaks: [1050 1808]
Rピーク数: 2
Rピーク間隔: [758]


### P,QRS,Tを含めた検出

In [46]:
def detect_ecg_waves(signal: np.ndarray, fs: int = 1000) -> dict:
    """
    ECG波形からP波・QRS波・T波のピーク位置を検出する関数

    Parameters:
        signal (np.ndarray): NaN除去済みのECG波形
        fs (int): サンプリング周波数（Hz）

    Returns:
        dict: 構成波のピーク位置（各波ごとにnp.ndarray）
    """
    if len(signal) < 100:
        raise ValueError("信号が短すぎます。最低100サンプル以上必要です。")

    # Rピーク検出
    signals, info = nk.ecg_peaks(signal, sampling_rate=fs)
    rpeaks = pd.Series(info["ECG_R_Peaks"]).dropna().astype(int).values

    # 構成波検出
    _, waves = nk.ecg_delineate(signal, rpeaks=rpeaks, sampling_rate=fs, method="dwt")

    # 各波のピーク位置を抽出
    wave_peaks = {}
    for wave in ["ECG_P_Peaks", "ECG_Q_Peaks", "ECG_R_Peaks", "ECG_S_Peaks", "ECG_T_Peaks"]:
        wave_peaks[wave] = pd.Series(waves.get(wave, [])).dropna().astype(int).values

    return wave_peaks

In [54]:
#- Rピーク数が 5 未満 → nk.ecg_delineate() は失敗
#- Rピーク間隔がすべて同じ → 周期性が強すぎて失敗
signal = extract_lead_signal(ecg_df, lead="I")
#無理やり結合
signal =np.tile(signal, 3)

result = detect_ecg_waves(signal, fs=1000)
result

{'ECG_P_Peaks': array([ 937, 1698, 2651, 3437, 4198, 5152, 5937, 6698]),
 'ECG_Q_Peaks': array([1025, 1786, 2776, 3525, 4286, 5276, 6025, 6786]),
 'ECG_R_Peaks': array([], dtype=int64),
 'ECG_S_Peaks': array([1118, 1865, 2892, 3618, 4365, 5392, 6118, 6865]),
 'ECG_T_Peaks': array([1300, 2060, 3050, 3800, 4560, 5550, 6300, 7060])}

In [56]:
# plotを行う関数
def plot_ecg_waves(signal: np.ndarray, wave_peaks: dict, fs: int = 1000, title: str = "ECG Wave Delineation"):
    """
    ECG波形と構成波（P, Q, R, S, T）を色分けして可視化する関数

    Parameters:
        signal (np.ndarray): ECG波形
        wave_peaks (dict): detect_ecg_waves() の出力
        fs (int): サンプリング周波数（Hz）
        title (str): プロットタイトル
    """
    time_axis = np.arange(len(signal)) / fs
    plt.figure(figsize=(15, 4))
    plt.plot(time_axis, signal, label="ECG Signal", color="black")

    wave_colors = {
        "ECG_P_Peaks": "blue",
        "ECG_Q_Peaks": "purple",
        "ECG_R_Peaks": "red",
        "ECG_S_Peaks": "orange",
        "ECG_T_Peaks": "green"
    }

    for wave, color in wave_colors.items():
        peaks = wave_peaks.get(wave, [])
        if isinstance(peaks, (np.ndarray, list)) and len(peaks) > 0:
            plt.scatter(time_axis[peaks], signal[peaks], color=color,
                        label=wave.replace("ECG_", "").replace("_Peaks", ""), s=40)

    plt.title(title)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()